In [1]:
import os
import pandas as pd

In [ ]:
dataset_path = "data_set"

# Function to check if the three CSV files have the same number of rows (check if the sync is okay)
def check_csv_files(folder_path):
    print("Checking folder", folder_path)
    
    # Remove unwanted files if they exist
    for unwanted_file in ["video.mp4", "eyetracker.csv"]:
        unwanted_path = os.path.join(folder_path, unwanted_file)
        if os.path.exists(unwanted_path):
            try:
                os.remove(unwanted_path)
                # print(f"Deleted {unwanted_file}")
            except Exception as e:
                print(f"Failed to delete {unwanted_file}: {e}")

    try:
        insoles_df = pd.read_csv(os.path.join(folder_path, "insoles.csv"))
        labels_df = pd.read_csv(os.path.join(folder_path, "labels.csv"))
        xsens_df = pd.read_csv(os.path.join(folder_path, "xsens.csv"))

        rows_insoles = len(insoles_df)
        rows_labels = len(labels_df)
        rows_xsens = len(xsens_df)

        if rows_insoles == rows_labels and rows_insoles == rows_xsens:
            merged_df = pd.concat([insoles_df, labels_df, xsens_df], axis=1)
            merged_df = merged_df.loc[~merged_df['walk_mode'].isin(['pavement_down', 'pavement_up'])]  # Filter specific walk modes
            merged_df = merged_df.dropna()
            merged_df.to_csv(os.path.join(folder_path, "merged.csv"), index=False)
            return True
        return False
    except Exception as e:
        print(f"Error reading CSV files in {folder_path}: {e}")
        return False

In [ ]:
for course_folder in os.listdir(dataset_path):
    course_folder_path = os.path.join(dataset_path, course_folder)
    if os.path.isdir(course_folder_path):
        for subfolder in os.listdir(course_folder_path):
            subfolder_path = os.path.join(course_folder_path, subfolder)
            if os.path.isdir(subfolder_path):
                if not check_csv_files(subfolder_path):
                    print(f"Mismatch in number of rows in folder: {subfolder_path}")
print("Dataset merged successefully")